In [ ]:
assay_data_path = None  

In [ ]:
# ------------------------------ imports -------------------------------------- #
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter, defaultdict
from joblib import Parallel, delayed
from scipy.integrate import trapezoid
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc,
    accuracy_score, f1_score, confusion_matrix,
    average_precision_score, brier_score_loss, 
    log_loss)
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy
import xgboost as xgb
from pathlib import Path
import json
import re
import joblib
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import shap
from imblearn.pipeline import Pipeline as ImbPipeline


# ------------------------------ config --------------------------------------- #
OUT_BASE = Path("../output")
OUT_BASE.mkdir(parents=True, exist_ok=True)

fold_importances, fold_predictions = [], []
fold_metrics_detailed = []  
fold_class_distributions = []  
fold_feature_selection_stats = [] 

N_SPLITS        = 3
RANDOM_STATE    = 21
N_JOBS          = 40
STABILITY_RUNS  = 5
STABILITY_MIN_FRAC = 0.6 


# ------------------------------ data ----------------------------------------- #
chemical_httr_assay_aggregated = pd.read_feather("../../HTTr_and_chemical_data/output/chemical_httr_assay_aggregated_alternative.feather")

# update colnames
httr_cols     = [col for col in chemical_httr_assay_aggregated.columns if col.endswith('_max') or col.endswith('_dose_at_max') or col.endswith('_AUC_neg')]
chemical_cols = [col for col in chemical_httr_assay_aggregated.columns if col.startswith('MACCS_')]
assay_cols    = [col for col in chemical_httr_assay_aggregated.columns if col.startswith('TOX21_')]
metadata_cols = [col for col in chemical_httr_assay_aggregated.columns if col not in httr_cols + chemical_cols + assay_cols]

selected_assay = Path(assay_data_path).stem 

if 'cell_type' in chemical_httr_assay_aggregated.columns:
    chemical_httr_assay_aggregated = chemical_httr_assay_aggregated.drop(columns=['cell_type'])

df_filtered = chemical_httr_assay_aggregated.dropna(subset=[selected_assay]).copy()

cols_to_drop = [c for c in df_filtered.columns if c.startswith("TOX21_") and c != selected_assay] + ['chnm']
df_filtered.drop(columns=cols_to_drop, inplace=True)

ASSAY_SAFE = re.sub(r"[^A-Za-z0-9_.-]+", "_", selected_assay)
RUN_DIR = OUT_BASE / ASSAY_SAFE
RUN_DIR.mkdir(parents=True, exist_ok=True)

IDs_TO_KEEP = [c for c in ['outcome_id','epa_sample_id'] if c in df_filtered.columns]

y_raw    = df_filtered[selected_assay].astype(int).values
groups   = df_filtered['outcome_id'].astype(str).values  # keep chemicals from leaking

y_strat = y_raw

pos = int((y_raw == 1).sum())
neg = int((y_raw == 0).sum())

print(f"\nTarget: \033[1m{selected_assay}\033[0m")
print(f"Samples: {len(df_filtered):,} | Positives: {pos:,} | Negatives: {neg:,}")
print(' ')

pos = int((y_raw == 1).sum())
neg = int((y_raw == 0).sum())

initial_stats = {
    'total_samples': len(chemical_httr_assay_aggregated),
    'samples_with_target': len(df_filtered),
    'n_httr_features': len(httr_cols),
    'n_chemical_features': len(chemical_cols),
    'n_assay_features': len(assay_cols),
    'target_positives': int((y_raw==1).sum()),
    'target_negatives': int((y_raw==0).sum()),
    'target_balance_ratio': min(int((y_raw==1).sum()), int((y_raw==0).sum())) / max(int((y_raw==1).sum()), int((y_raw==0).sum())),
    'unique_chemicals': df_filtered['outcome_id'].nunique() if 'outcome_id' in df_filtered.columns else None
}


# ------------------------------ CV setup ------------------------------------- #
cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_metrics  = []
feature_hits  = Counter()
features_used = Counter()
fold_selected_features = []
_fold_best_params = [] 
fold_conf_mats = []

start_all = datetime.now()

# OOF dummies
oof_proba = np.full(len(df_filtered), np.nan)
oof_true  = np.zeros(len(df_filtered), dtype=int)


# ------------------------------ CV start ------------------------------------- #
for fold, (train_idx, test_idx) in enumerate(cv.split(df_filtered, y_strat, groups), start=1):

    train = df_filtered.iloc[train_idx].reset_index(drop=True)
    test  = df_filtered.iloc[test_idx].reset_index(drop=True)
    
    y_train = train[selected_assay].astype(int).values
    y_test  = test[selected_assay].astype(int).values

    # safety check (should not be the case)
    if len(np.unique(y_train)) < 2:
        print(f"[Fold {fold}] skipped (train has a single class).")
        continue

    # ----------------------- Prepare data for Boruta ------------------------- #
    meta_cols = [c for c in ['outcome_id', 'epa_sample_id'] if c in train.columns]

    X_train_df = train.drop(columns=[selected_assay]).copy()
    X_train_df = X_train_df.drop(columns=meta_cols)

    X_test_df = test.drop(columns=[selected_assay] + meta_cols).copy()

    maccs_cols = [c for c in X_train_df.columns if c.startswith("MACCS_")]
    
    if maccs_cols:
        X_train_df[maccs_cols] = X_train_df[maccs_cols].astype("int8")
        if len(X_test_df) > 0:
            X_test_df[maccs_cols] = X_test_df[maccs_cols].astype("int8")

    fold_pos = int((y_train == 1).sum())
    fold_neg = int((y_train == 0).sum())
    fold_total = fold_pos + fold_neg
    fold_minority_ratio = min(fold_pos, fold_neg) / max(fold_pos, fold_neg) if max(fold_pos, fold_neg) > 0 else 1.0


    # ----------------------- Boruta on HTTr only (undersample + stability) --- #
    httr_cols = [col for col in X_train_df.columns
                 if col.endswith('_max') or col.endswith('_dose_at_max') or col.endswith('_AUC_neg')]
    
    X_train_httr_full = X_train_df[httr_cols].values
    rf_for_boruta = RandomForestClassifier(
        n_jobs=N_JOBS,
        n_estimators=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
    
    # stability over multiple random undersamples (negatives down to #positives)
    hits_counter = Counter()
    valid_runs = 0
    pos_idx_all = np.where(y_train == 1)[0]
    neg_idx_all = np.where(y_train == 0)[0]
    for r in range(STABILITY_RUNS):
        if len(neg_idx_all) > len(pos_idx_all) and len(pos_idx_all) > 0:
            rs = np.random.RandomState(RANDOM_STATE + r)
            neg_sel = rs.choice(neg_idx_all, size=len(pos_idx_all), replace=False)
            sel_idx = np.concatenate([pos_idx_all, neg_sel])
        else:
            sel_idx = np.arange(len(y_train))

        X_train_httr_boruta = X_train_df.iloc[sel_idx][httr_cols].values
        y_train_boruta = y_train[sel_idx]

        boruta = BorutaPy(
            estimator=rf_for_boruta,
            n_estimators='auto',
            perc=100,
            verbose=0,
            random_state=RANDOM_STATE + r
        )
        try:
            boruta.fit(X_train_httr_boruta, y_train_boruta)
            selected_mask_run = boruta.support_
            for f, keep in zip(httr_cols, selected_mask_run):
                if keep:
                    hits_counter[f] += 1
            valid_runs += 1
        except Exception as _e:
            continue
    
    # keep features selected in >= STABILITY_MIN_FRAC of valid runs
    threshold_hits = int(np.ceil(STABILITY_MIN_FRAC * max(valid_runs, 1)))
    selected_httr_features = [f for f, h in hits_counter.items() if h >= threshold_hits]
    
    # fallback if stability keeps nothing
    fallback_used = False
    if len(selected_httr_features) == 0:
        rf_for_boruta.fit(X_train_httr_full, y_train)
        importances = rf_for_boruta.feature_importances_
        order = np.argsort(importances)[::-1]
        top_k = max(25, min(200, int(np.sqrt(len(httr_cols)))))
        selected_httr_features = [httr_cols[i] for i in order[:top_k]]
        fallback_used = True
    
    fold_selected_features.append(selected_httr_features)
    feature_hits.update(selected_httr_features)


    # ----------------------- Prepare pipeline with SMOTE + XGB -------------- #
    feature_cols = chemical_cols + selected_httr_features 
    features_used.update(feature_cols)

    X_train_selected = X_train_df[feature_cols]
    X_test_full = X_test_df[feature_cols] if len(X_test_df) > 0 else test[feature_cols]

    pre_smote_pos = fold_pos
    pre_smote_neg = fold_neg

    categorical_idx = [feature_cols.index(c) for c in feature_cols if c in maccs_cols]  # only MACCS categorical

    # Create SMOTE-XGB pipeline to prevent data leakage
    smote = SMOTENC(
        categorical_features=categorical_idx,
        sampling_strategy='auto', 
        random_state=RANDOM_STATE,
    )

    # For reporting purposes, apply SMOTE once to get post-SMOTE stats
    X_resampled_arr, y_resampled = smote.fit_resample(X_train_selected, y_train)
    post_smote_pos = int((y_resampled == 1).sum())
    post_smote_neg = int((y_resampled == 0).sum())
    
    fold_class_distributions.append({
        'fold': fold,
        'train_samples': len(y_train),
        'test_samples': len(y_test),
        'train_pos_pre_smote': pre_smote_pos,
        'train_neg_pre_smote': pre_smote_neg,
        'train_pos_post_smote': post_smote_pos,
        'train_neg_post_smote': post_smote_neg,
        'test_pos': int((y_test == 1).sum()),
        'test_neg': int((y_test == 0).sum()),
        'smote_applied': True,
        'fold_minority_ratio_pre_smote': fold_minority_ratio
    })

    # ----------------------- XGB + RandomizedSearchCV with Pipeline ---------- #
    # Since SMOTE balances classes, we don't need scale_pos_weight
    base_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=RANDOM_STATE,
        n_jobs=1,
        n_estimators=1000,
        learning_rate=0.1,
        tree_method='hist'
    )

    # Create pipeline with SMOTE + XGB
    pipeline = ImbPipeline([
        ('smote', SMOTENC(
            categorical_features=categorical_idx,
            sampling_strategy='auto',
            random_state=RANDOM_STATE
        )),
        ('classifier', base_model)
    ])

    param_distributions = {
        'classifier__max_depth': [3, 4, 5, 6, 7, 8],
        'classifier__colsample_bytree': [0.4, 0.6, 0.8, 1.0],
        'classifier__reg_alpha': [0, 1e-3, 1e-2, 1e-1, 1, 5],
        'classifier__reg_lambda': [0.5, 1, 2, 5, 10],
        'classifier__learning_rate': [0.02, 0.05, 0.1],
        'classifier__n_estimators': [400, 600, 800, 1000, 1200]
    }

    # Use StratifiedGroupKFold for inner CV to prevent chemical leakage
    train_groups = train['outcome_id'].astype(str).values
    inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    tuner = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=50,                    
        scoring='average_precision', 
        cv=inner_cv,  # Pass the CV object, not the split method
        verbose=0,
        n_jobs=N_JOBS,
        refit=True,
        random_state=RANDOM_STATE
    )

    tuner.fit(X_train_selected, y_train, groups=train_groups)  # Pass groups for StratifiedGroupKFold
    model = tuner.best_estimator_
    _fold_best_params.append({'fold': fold, **{k.replace('classifier__', ''): v for k, v in tuner.best_params_.items() if k.startswith('classifier__')}})

    # ----------------------- Evaluate (store OOF) ---------------------------- #
    y_proba = model.predict_proba(X_test_full)[:, 1]

    # Get the actual trained XGB model for SHAP
    xgb_model = model.named_steps['classifier']


    # ----------------------- SHAP (per-fold test) ---------------------------- #
    shap_dir = RUN_DIR / 'shap'
    shap_dir.mkdir(parents=True, exist_ok=True)
    try:
        # Use the SMOTE-transformed training data for SHAP background
        X_resampled_df = pd.DataFrame(X_resampled_arr, columns=feature_cols)
        explainer = shap.Explainer(xgb_model, X_resampled_df, algorithm="tree", feature_names=feature_cols, model_output="probability")
        shap_explanation = explainer(X_test_full)
        shap_values_test = shap_explanation.values
        base_values_test = shap_explanation.base_values
    except Exception:
        try:
            explainer = shap.TreeExplainer(xgb_model, data=X_resampled_df, feature_perturbation="interventional", model_output="probability")
            shap_values_test = explainer.shap_values(X_test_full)
            base_values_test = explainer.expected_value
            if np.isscalar(base_values_test):
                base_values_test = np.repeat(base_values_test, len(X_test_full))
        except Exception:
            base_values_test = np.full(len(X_test_full), np.nan, dtype=float)
            shap_values_test = np.zeros((len(X_test_full), len(feature_cols)))

    shap_cols = [f"SHAP_{c}" for c in feature_cols]
    shap_df_fold = pd.DataFrame(shap_values_test, columns=shap_cols)
    meta_df = test[IDs_TO_KEEP].copy() if len(IDs_TO_KEEP) else pd.DataFrame(index=np.arange(len(test)))
    meta_df['fold'] = fold
    meta_df['y_true'] = y_test
    meta_df['y_proba'] = y_proba
    meta_df['base_value'] = base_values_test
    feat_vals_df = X_test_full.reset_index(drop=True).copy()
    feat_vals_df.columns = feature_cols
    shap_out_fold = pd.concat([meta_df.reset_index(drop=True), feat_vals_df, shap_df_fold], axis=1)
    (shap_dir / 'per_fold').mkdir(exist_ok=True, parents=True)
    shap_out_path = shap_dir / 'per_fold' / f'shap_test_fold_{fold}.feather'
    shap_out_fold.to_feather(shap_out_path)
    mean_abs_shap = np.abs(shap_values_test).mean(axis=0)
    shap_rank_fold = pd.DataFrame({'fold': fold,'feature': feature_cols,'mean_abs_shap': mean_abs_shap}).sort_values('mean_abs_shap', ascending=False)
    shap_rank_path = shap_dir / 'per_fold' / f'shap_rank_fold_{fold}.csv'
    shap_rank_fold.to_csv(shap_rank_path, index=False)

    # store OOF predictions for global threshold selection
    oof_proba[test_idx] = y_proba
    oof_true[test_idx]  = y_test


    # -------- threshold-free metrics --------
    try:
        roc = roc_auc_score(y_test, y_proba)
    except ValueError:
        roc = np.nan

    precision, recall, thr = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision) if (len(recall) > 1 and len(precision) > 1) else np.nan

    try:
        ap = average_precision_score(y_test, y_proba)
    except ValueError:
        ap = np.nan

    brier = brier_score_loss(y_test, y_proba)
    try:
        ll = log_loss(y_test, y_proba)
    except ValueError:
        ll = np.nan


    # -------- per-fold optimal F1 threshold --------
    if thr.size > 0:
        # precision and recall arrays are 1 element longer than thresholds
        f1s = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-12)
        best_idx = int(np.nanargmax(f1s))
        fold_opt_threshold = float(thr[best_idx])

        y_pred_opt = (y_proba >= fold_opt_threshold).astype(int)
        acc_opt = accuracy_score(y_test, y_pred_opt)
        f1_opt  = f1_score(y_test, y_pred_opt, zero_division=0)
        cm_opt  = confusion_matrix(y_test, y_pred_opt, labels=[0, 1])
    else:
        fold_opt_threshold = 0.5
        y_pred_opt = (y_proba >= 0.5).astype(int) 
        acc_opt = np.nan
        f1_opt  = np.nan
        cm_opt  = confusion_matrix(y_test, y_pred_opt, labels=[0, 1])

    fold_conf_mats.append(cm_opt)

    fold_metrics_detailed.append({
        'fold': fold,
        'n_train': len(y_resampled),  # Use y_resampled since that's the actual training size after SMOTE
        'n_test': len(y_test),
        'train_pos': post_smote_pos,
        'train_neg': post_smote_neg,
        'test_pos': int((y_test == 1).sum()),
        'test_neg': int((y_test == 0).sum()),
        'roc_auc': roc,
        'pr_auc': pr_auc,
        'average_precision': ap,
        'brier_score': brier,
        'log_loss': ll,
        'optimal_threshold': fold_opt_threshold,
        'accuracy_at_optimal': acc_opt,
        'f1_at_optimal': f1_opt,
        'tn': int(cm_opt[0,0]),
        'fp': int(cm_opt[0,1]),
        'fn': int(cm_opt[1,0]),
        'tp': int(cm_opt[1,1]),
        'precision_at_optimal': int(cm_opt[1,1]) / (int(cm_opt[1,1]) + int(cm_opt[0,1])) if (int(cm_opt[1,1]) + int(cm_opt[0,1])) > 0 else 0,
        'recall_at_optimal': int(cm_opt[1,1]) / (int(cm_opt[1,1]) + int(cm_opt[1,0])) if (int(cm_opt[1,1]) + int(cm_opt[1,0])) > 0 else 0,
        'specificity_at_optimal': int(cm_opt[0,0]) / (int(cm_opt[0,0]) + int(cm_opt[0,1])) if (int(cm_opt[0,0]) + int(cm_opt[0,1])) > 0 else 0,
        'scale_pos_weight': None,  # Not used with SMOTE pipeline
        'httr_features_selected': len(selected_httr_features),
        'best_params': json.dumps({k.replace('classifier__', ''): v for k, v in tuner.best_params_.items() if k.startswith('classifier__')})
    })


    # ----------------------- Logging per-fold --------------------------------
    try:
        # Extract feature importances from the XGB model in the pipeline
        if hasattr(xgb_model, 'feature_importances_') and len(xgb_model.feature_importances_) == len(feature_cols):
            imp_df = pd.DataFrame({
                'fold': fold,
                'feature': feature_cols,
                'importance': xgb_model.feature_importances_
            })
            fold_importances.append(imp_df)
    except Exception as _e:
        pass

    try:
        pred_df_fold = test[IDs_TO_KEEP].copy() if len(IDs_TO_KEEP) else pd.DataFrame(index=np.arange(len(test)))
        pred_df_fold['fold'] = fold

        # still store raw proba + preds so global threshold step works later
        pred_df_fold['threshold'] = fold_opt_threshold  # you can store the opt threshold here
        pred_df_fold['y_true'] = y_test
        pred_df_fold['y_proba'] = y_proba
        pred_df_fold['y_pred']  = y_pred_opt

        fold_predictions.append(pred_df_fold)
    except Exception as _e:
        pass

    try:
        (RUN_DIR / 'models').mkdir(parents=True, exist_ok=True)
        joblib.dump(model, RUN_DIR / 'models' / f'model_fold_{fold}.joblib')
    except Exception as _e:
        pass

    cmopt_str = f"[{cm_opt.ravel()[0]},{cm_opt.ravel()[1]};{cm_opt.ravel()[2]},{cm_opt.ravel()[3]}]"
    print(
        f"\033[1m[Fold {fold}/{N_SPLITS}]\033[0m "
        f"HTTr selected: {len(selected_httr_features):>4} | "
        f"ROC AUC: {roc:.3f} | PR AUC: {pr_auc:.3f} | AP: {ap:.3f} | "
        f"Brier: {brier:.3f} | LogLoss: {ll:.3f} | "
        f"ACC: {acc_opt:.3f} | F1: {f1_opt:.3f} | CM: {cmopt_str} | thr: {fold_opt_threshold:.3f}"
    )

end_all = datetime.now()

# ------------------------------ Global threshold (after CV) ------------------ #
mask = ~np.isnan(oof_proba)
y_true_oof = oof_true[mask]
y_prob_oof = oof_proba[mask]

if len(y_true_oof) == 0:
    global_threshold = 0.5
    print("No valid OOF predictions found; using default threshold 0.5")
else:
    prec_oof, rec_oof, thr_oof = precision_recall_curve(y_true_oof, y_prob_oof)
    if len(thr_oof) == 0:
        global_threshold = 0.5
        print("Unable to compute PR thresholds; using default threshold 0.5")
    else:
        # precision and recall arrays are 1 element longer than thresholds
        f1s = (2 * prec_oof[:-1] * rec_oof[:-1]) / (prec_oof[:-1] + rec_oof[:-1] + 1e-12)
        best_idx = int(np.nanargmax(f1s))
        global_threshold = float(thr_oof[best_idx])

    y_pred_oof = (y_prob_oof >= global_threshold).astype(int)
    oof_roc = roc_auc_score(y_true_oof, y_prob_oof) if len(np.unique(y_true_oof)) > 1 else float('nan')
    oof_pr  = auc(rec_oof, prec_oof) if (len(rec_oof) > 1 and len(prec_oof) > 1) else float('nan')
    oof_acc = accuracy_score(y_true_oof, y_pred_oof)
    oof_f1  = f1_score(y_true_oof, y_pred_oof, zero_division=0)
    oof_cm  = confusion_matrix(y_true_oof, y_pred_oof, labels=[0,1])

    print(f"\nOOF chosen threshold (max F1): {global_threshold:.4f}")
    print(f"OOF ROC AUC: {oof_roc:.3f} | OOF PR AUC: {oof_pr:.3f} | "
          f"OOF ACC: {oof_acc:.3f} | OOF F1: {oof_f1:.3f}")
    print(f"OOF Confusion Matrix: {oof_cm.tolist()}")

# ------------------------------ SHAP aggregation across folds ---------------- #
try:
    per_fold_dir = RUN_DIR / 'shap' / 'per_fold'
    if per_fold_dir.exists():
        shap_files = sorted(per_fold_dir.glob('shap_test_fold_*.feather'))
        if shap_files:
            oof_shap_df = pd.concat([pd.read_feather(p) for p in shap_files], axis=0, ignore_index=True)
            oof_shap_path = RUN_DIR / 'shap' / 'oof_shap.feather'
            oof_shap_df.to_feather(oof_shap_path)

            shap_cols_all = [c for c in oof_shap_df.columns if c.startswith('SHAP_')]
            feature_names_from_shap = [c.replace('SHAP_', '') for c in shap_cols_all]
            mean_abs = oof_shap_df[shap_cols_all].abs().mean(axis=0).values
            global_rank_df = pd.DataFrame({
                'feature': feature_names_from_shap,
                'mean_abs_shap_oof': mean_abs
            }).sort_values('mean_abs_shap_oof', ascending=False)

            try:
                feat_hits_df = pd.DataFrame.from_dict(feature_hits, orient='index').reset_index()
                feat_hits_df.columns = ['feature', 'hit_count']
                global_rank_df = global_rank_df.merge(feat_hits_df, on='feature', how='left')
            except Exception:
                pass

            global_rank_df['feature_type'] = global_rank_df['feature'].apply(
                lambda x: 'httr' if any(x.endswith(suffix) for suffix in ['_max', '_dose_at_max', '_AUC_neg'])
                else 'chemical' if x.startswith('MACCS_') else 'other'
            )

            (RUN_DIR / 'shap').mkdir(exist_ok=True, parents=True)
            global_rank_df.to_csv(RUN_DIR / 'shap' / 'shap_global_rank_oof.csv', index=False)
            global_rank_df.head(50).to_csv(RUN_DIR / 'shap' / 'shap_top50_oof.csv', index=False)

            rank_files = sorted(per_fold_dir.glob('shap_rank_fold_*.csv'))
            if rank_files:
                all_ranks_df = pd.concat([pd.read_csv(p) for p in rank_files], axis=0, ignore_index=True)
                all_ranks_df.to_csv(RUN_DIR / 'shap' / 'shap_rank_all_folds.csv', index=False)
except Exception as _e:
    pass

# ------------------------------ Aggregate + persist artifacts ---------------- #
# Feature selection summary
try:
    feat_hits_df = pd.DataFrame.from_dict(feature_hits, orient='index').reset_index()
    feat_hits_df.columns = ['feature', 'hit_count']
    feat_hits_df.sort_values('hit_count', ascending=False, inplace=True)
    feat_hits_df.to_csv(RUN_DIR / 'feature_hits.csv', index=False)
except Exception as _e:
    pass

# Importances
try:
    if len(fold_importances):
        importances_df = pd.concat(fold_importances, axis=0, ignore_index=True)
        importances_df.to_csv(RUN_DIR / 'fold_feature_importances.csv', index=False)
except Exception as _e:
    pass

# Predictions
try:
    if len(fold_predictions):
        preds_df = pd.concat(fold_predictions, axis=0, ignore_index=True)
        preds_df['threshold'] = global_threshold  # reflect final threshold
        preds_df.to_csv(RUN_DIR / 'fold_predictions.csv', index=False)
except Exception as _e:
    pass

# Save detailed fold metrics
try:
    if len(fold_metrics_detailed):
        metrics_df = pd.DataFrame(fold_metrics_detailed)
        metrics_df.to_csv(RUN_DIR / 'fold_metrics_detailed.csv', index=False)
except Exception as _e:
    pass

# Save class distributions per fold
try:
    if len(fold_class_distributions):
        class_dist_df = pd.DataFrame(fold_class_distributions)
        class_dist_df.to_csv(RUN_DIR / 'class_distributions.csv', index=False)
except Exception as _e:
    pass

# Save feature selection statistics per fold
try:
    if len(fold_feature_selection_stats):
        feat_sel_df = pd.DataFrame(fold_feature_selection_stats)
        feat_sel_df.to_csv(RUN_DIR / 'feature_selection_stats.csv', index=False)
except Exception as _e:
    pass

# Save overall run summary
try:
    if len(fold_metrics_detailed):
        metrics_df = pd.DataFrame(fold_metrics_detailed)
        cv_summary = {
            'target_assay': selected_assay,
            'n_folds': N_SPLITS,
            'random_state': RANDOM_STATE,
            'global_threshold': global_threshold,
            'cv_roc_auc_mean': float(metrics_df['roc_auc'].mean()),
            'cv_roc_auc_std': float(metrics_df['roc_auc'].std()),
            'cv_pr_auc_mean': float(metrics_df['pr_auc'].mean()),
            'cv_pr_auc_std': float(metrics_df['pr_auc'].std()),
            'cv_average_precision_mean': float(metrics_df['average_precision'].mean()),
            'cv_average_precision_std': float(metrics_df['average_precision'].std()),
            'cv_f1_mean': float(metrics_df['f1_at_optimal'].mean()),
            'cv_f1_std': float(metrics_df['f1_at_optimal'].std()),
            'cv_accuracy_mean': float(metrics_df['accuracy_at_optimal'].mean()),
            'cv_accuracy_std': float(metrics_df['accuracy_at_optimal'].std()),
            'cv_precision_mean': float(metrics_df['precision_at_optimal'].mean()),
            'cv_precision_std': float(metrics_df['precision_at_optimal'].std()),
            'cv_recall_mean': float(metrics_df['recall_at_optimal'].mean()),
            'cv_recall_std': float(metrics_df['recall_at_optimal'].std()),
            'cv_specificity_mean': float(metrics_df['specificity_at_optimal'].mean()),
            'cv_specificity_std': float(metrics_df['specificity_at_optimal'].std()),
            'cv_brier_score_mean': float(metrics_df['brier_score'].mean()),
            'cv_brier_score_std': float(metrics_df['brier_score'].std()),
            'cv_log_loss_mean': float(metrics_df['log_loss'].mean()),
            'cv_log_loss_std': float(metrics_df['log_loss'].std()),
            'oof_roc_auc': float(oof_roc) if 'oof_roc' in locals() and not np.isnan(oof_roc) else None,
            'oof_pr_auc': float(oof_pr) if 'oof_pr' in locals() and not np.isnan(oof_pr) else None,
            'oof_accuracy': float(oof_acc) if 'oof_acc' in locals() else None,
            'oof_f1': float(oof_f1) if 'oof_f1' in locals() else None,
            **initial_stats,
            'run_start_time': start_all.isoformat(),
            'run_end_time': end_all.isoformat(),
            'run_duration_seconds': (end_all - start_all).total_seconds()
        }
    else:
        cv_summary = {
            'target_assay': selected_assay,
            'n_folds': N_SPLITS,
            'random_state': RANDOM_STATE,
            'global_threshold': global_threshold,
            **initial_stats,
            'run_start_time': start_all.isoformat(),
            'run_end_time': end_all.isoformat(),
            'run_duration_seconds': (end_all - start_all).total_seconds(),
            'note': 'No valid folds completed'
        }
    
    summary_df = pd.DataFrame([cv_summary])
    
    with open(RUN_DIR / 'run_summary.json', 'w') as f:
        json.dump(cv_summary, f, indent=2)
except Exception as _e:
    pass

# Save aggregated feature importance statistics
try:
    if len(fold_importances):
        importances_df = pd.concat(fold_importances, axis=0, ignore_index=True)
        
        # Calculate importance statistics across folds
        importance_stats = importances_df.groupby('feature')['importance'].agg([
            'count', 'mean', 'std', 'min', 'max', 'median'
        ]).reset_index()
        importance_stats.columns = ['feature', 'fold_count', 'importance_mean', 
                                  'importance_std', 'importance_min', 'importance_max', 'importance_median']
        importance_stats = importance_stats.sort_values('importance_mean', ascending=False)
        
        # Add feature type information
        importance_stats['feature_type'] = importance_stats['feature'].apply(
            lambda x: 'httr' if any(x.endswith(suffix) for suffix in ['_max', '_dose_at_max', '_AUC_neg']) 
                     else 'chemical' if x.startswith('MACCS_') 
                     else 'other'
        )
        
        importance_stats.to_csv(RUN_DIR / 'feature_importance_summary.csv', index=False)
except Exception as _e:
    pass

# Save selected features per fold for reproducibility
try:
    if len(fold_selected_features):
        fold_features_data = []
        for fold_idx, features in enumerate(fold_selected_features, 1):
            for feature in features:
                fold_features_data.append({
                    'fold': fold_idx,
                    'feature': feature,
                    'feature_type': 'httr' if any(feature.endswith(suffix) for suffix in ['_max', '_dose_at_max', '_AUC_neg']) else 'other'
                })
        
        fold_features_df = pd.DataFrame(fold_features_data)
        fold_features_df.to_csv(RUN_DIR / 'selected_features_per_fold.csv', index=False)
except Exception as _e:
    pass

end_all = datetime.now()
print(f"\nRun directory: {RUN_DIR}")
print(f"Started: {start_all} | Ended: {end_all}")



Target: TOX21_PR_BLA_Antagonist_ratio
Samples: 2,006 | Positives: 673 | Negatives: 1,333
 


 96%|=================== | 391/406 [00:19<00:00]       

[Fold 1/5] HTTr selected:  161 | ROC AUC: 0.946 | PR AUC: 0.915 | AP: 0.915 | Brier: 0.081 | LogLoss: 0.326 | ACC: 0.906 | F1: 0.857 | CM: [254,19;19,114] | thr: 0.462


 97%|=================== | 399/411 [00:26<00:00]       

[Fold 2/5] HTTr selected:  156 | ROC AUC: 0.959 | PR AUC: 0.949 | AP: 0.950 | Brier: 0.082 | LogLoss: 0.264 | ACC: 0.888 | F1: 0.874 | CM: [206,29;17,159] | thr: 0.259


...

[Fold 3/5] HTTr selected:  214 | ROC AUC: 0.886 | PR AUC: 0.798 | AP: 0.799 | Brier: 0.140 | LogLoss: 0.576 | ACC: 0.849 | F1: 0.781 | CM: [226,39;20,105] | thr: 0.195


 96%|=================== | 392/407 [00:25<00:00]       

[Fold 4/5] HTTr selected:  201 | ROC AUC: 0.922 | PR AUC: 0.835 | AP: 0.837 | Brier: 0.100 | LogLoss: 0.347 | ACC: 0.865 | F1: 0.789 | CM: [249,41;14,103] | thr: 0.368


 98%|===================| 383/392 [00:24<00:00]        

[Fold 5/5] HTTr selected:  156 | ROC AUC: 0.954 | PR AUC: 0.899 | AP: 0.899 | Brier: 0.085 | LogLoss: 0.287 | ACC: 0.893 | F1: 0.838 | CM: [241,29;13,109] | thr: 0.287

OOF chosen threshold (max F1): 0.2464
OOF ROC AUC: 0.931 | OOF PR AUC: 0.879 | OOF ACC: 0.868 | OOF F1: 0.817
OOF Confusion Matrix: [[1151, 182], [83, 590]]

Run directory: output/papermill/TOX21_PR_BLA_Antagonist_ratio
Started: 2025-09-12 16:41:41.540751 | Ended: 2025-09-12 17:49:22.277983


<br> 

#### **02** - RESULTS